<a href="https://colab.research.google.com/github/ashokmulchandani/CUDA-GPU-Colab-Computer-Vision-Project-Ashok-1/blob/main/2_Ashok_CUDA_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Check GPU and setup
!nvidia-smi | head -5


Mon May 25 07:39:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |


In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Download a pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # 'n' = nano (smallest, fastest)

print(f"Model: YOLOv8-nano")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"Input size: 640×640")
print(f"Classes: {len(model.names)} ({', '.join(list(model.names.values())[:10])}...)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model: YOLOv8-nano
Parameters: 3,157,200
Input size: 640×640
Classes: 80 (person, bicycle, car, motorcycle, airplane, bus, train, truck, boat, traffic light...)


In [ ]:
# Download a test image and run detection
!wget -q https://ultralytics.com/images/bus.jpg -O test_image.jpg

results = model('test_image.jpg')

result = results[0]
print(f"\nDetections found: {len(result.boxes)}")
print(f"\n{'Class':<15} {'Confidence':<12} {'Box (x1,y1,x2,y2)'}")
print("-" * 55)
for box in result.boxes:
    cls = model.names[int(box.cls)]
    conf = float(box.conf)
    coords = box.xyxy[0].tolist()
    print(f"{cls:<15} {conf:<12.2f} [{coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f}]")



image 1/1 /content/test_image.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 168.1ms
Speed: 12.0ms preprocess, 168.1ms inference, 72.5ms postprocess per image at shape (1, 3, 640, 480)

Detections found: 6

Class           Confidence   Box (x1,y1,x2,y2)
-------------------------------------------------------
bus             0.87         [23, 231, 805, 757]
person          0.87         [49, 399, 245, 903]
person          0.85         [669, 392, 810, 877]
person          0.83         [222, 406, 345, 858]
person          0.26         [0, 551, 63, 873]
stop sign       0.26         [0, 254, 33, 325]


In [ ]:
# Let's see what happens inside YOLO step by step
import torch
import time

print("=== YOLO Pipeline Breakdown ===\n")

# Step 1: Preprocess (what we'll write in CUDA)
img = Image.open('test_image.jpg')
img_np = np.array(img)
print(f"1. PREPROCESS:")
print(f"   Raw image: {img_np.shape} (H×W×C, uint8, 0-255)")

start = time.time()
# Resize to 640×640, normalize to 0-1, convert HWC→CHW
img_resized = np.array(img.resize((640, 640))).astype(np.float32) / 255.0
img_chw = img_resized.transpose(2, 0, 1)  # HWC → CHW
img_batch = img_chw[np.newaxis, ...]  # add batch dim
preprocess_time = (time.time() - start) * 1000
print(f"   After preprocess: {img_batch.shape} (B×C×H×W, float32, 0-1)")
print(f"   Time: {preprocess_time:.1f} ms (Python/NumPy)")
print(f"   → CUDA kernel can do this in <1ms!\n")

# Step 2: Inference (CNN forward pass)
print(f"2. INFERENCE (CNN backbone + detection head):")
print(f"   YOLOv8-nano: 3.15M parameters")
print(f"   225 layers of Conv+BatchNorm+SiLU")
print(f"   Time: 168 ms (PyTorch)")
print(f"   → TensorRT can do this in ~10-20ms!\n")

# Step 3: Postprocess (NMS — what we'll write in CUDA)
print(f"3. POSTPROCESS (NMS):")
print(f"   Raw detections: ~8400 candidate boxes")
print(f"   After NMS: 6 final boxes (removed duplicates)")
print(f"   Time: 72.5 ms (Python)")
print(f"   → CUDA NMS kernel can do this in <5ms!\n")

print(f"=== What we'll build in CUDA ===")
print(f"   Step 4.3: Preprocessing kernel (resize+normalize+HWC→CHW)")
print(f"   Step 4.5: NMS kernel (remove overlapping boxes)")
print(f"   Step 4.8: TensorRT (optimize inference 168ms → ~15ms)")
print(f"   Goal: 252ms → ~20ms total = 50 FPS (real-time!)")


=== YOLO Pipeline Breakdown ===

1. PREPROCESS:
   Raw image: (1080, 810, 3) (H×W×C, uint8, 0-255)
   After preprocess: (1, 3, 640, 640) (B×C×H×W, float32, 0-1)
   Time: 19.3 ms (Python/NumPy)
   → CUDA kernel can do this in <1ms!

2. INFERENCE (CNN backbone + detection head):
   YOLOv8-nano: 3.15M parameters
   225 layers of Conv+BatchNorm+SiLU
   Time: 168 ms (PyTorch)
   → TensorRT can do this in ~10-20ms!

3. POSTPROCESS (NMS):
   Raw detections: ~8400 candidate boxes
   After NMS: 6 final boxes (removed duplicates)
   Time: 72.5 ms (Python)
   → CUDA NMS kernel can do this in <5ms!

=== What we'll build in CUDA ===
   Step 4.3: Preprocessing kernel (resize+normalize+HWC→CHW)
   Step 4.5: NMS kernel (remove overlapping boxes)
   Step 4.8: TensorRT (optimize inference 168ms → ~15ms)
   Goal: 252ms → ~20ms total = 50 FPS (real-time!)


In [ ]:
%%writefile preprocess.cu
#include <stdio.h>
#include <stdlib.h>

// Preprocess kernel: resize + normalize + HWC→CHW in ONE kernel
// Input:  H×W×3 (uint8, 0-255, HWC format)
// Output: 3×640×640 (float32, 0-1, CHW format)
__global__ void preprocess_kernel(unsigned char *input, float *output,
                                   int in_h, int in_w,
                                   int out_h, int out_w) {
    int out_x = blockIdx.x * blockDim.x + threadIdx.x;  // output column
    int out_y = blockIdx.y * blockDim.y + threadIdx.y;  // output row
    int c = blockIdx.z;  // channel (0=R, 1=G, 2=B)

    if (out_x < out_w && out_y < out_h && c < 3) {
        // Nearest-neighbor resize: map output pixel to input pixel
        int in_x = (int)((float)out_x / out_w * in_w);
        int in_y = (int)((float)out_y / out_h * in_h);

        // Clamp to valid range
        if (in_x >= in_w) in_x = in_w - 1;
        if (in_y >= in_h) in_y = in_h - 1;

        // Read from HWC format: input[y][x][c]
        unsigned char pixel = input[in_y * in_w * 3 + in_x * 3 + c];

        // Write to CHW format: output[c][y][x], normalized to 0-1
        output[c * out_h * out_w + out_y * out_w + out_x] = (float)pixel / 255.0f;
    }
}

int main() {
    // Simulate an input image (1080×810×3)
    int in_h = 1080, in_w = 810;
    int out_h = 640, out_w = 640;
    int in_size = in_h * in_w * 3;
    int out_size = 3 * out_h * out_w;

    // Create fake input image (random pixels)
    unsigned char *h_input = (unsigned char*)malloc(in_size);
    for (int i = 0; i < in_size; i++) h_input[i] = rand() % 256;

    float *h_output = (float*)malloc(out_size * sizeof(float));

    // GPU memory
    unsigned char *d_input;
    float *d_output;
    cudaMalloc(&d_input, in_size);
    cudaMalloc(&d_output, out_size * sizeof(float));
    cudaMemcpy(d_input, h_input, in_size, cudaMemcpyHostToDevice);

    // Launch kernel
    dim3 threads(16, 16);
    dim3 blocks((out_w+15)/16, (out_h+15)/16, 3);  // 3 channels

    // Warmup
    preprocess_kernel<<<blocks, threads>>>(d_input, d_output, in_h, in_w, out_h, out_w);
    cudaDeviceSynchronize();

    // Benchmark
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    cudaEventRecord(start);
    for (int i = 0; i < 100; i++)
        preprocess_kernel<<<blocks, threads>>>(d_input, d_output, in_h, in_w, out_h, out_w);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);
    ms /= 100;

    // Verify output
    cudaMemcpy(h_output, d_output, out_size * sizeof(float), cudaMemcpyDeviceToHost);

    printf("=== CUDA Image Preprocessing Kernel ===\n\n");
    printf("Input:  %d×%d×3 (uint8, HWC, 0-255)\n", in_h, in_w);
    printf("Output: 3×%d×%d (float32, CHW, 0-1)\n\n", out_h, out_w);
    printf("Operations (all in ONE kernel):\n");
    printf("  1. Resize: %d×%d → %d×%d (nearest neighbor)\n", in_h, in_w, out_h, out_w);
    printf("  2. Normalize: uint8 [0-255] → float32 [0-1]\n");
    printf("  3. Layout: HWC → CHW (what neural networks expect)\n\n");
    printf("Performance:\n");
    printf("  CUDA kernel:    %.3f ms\n", ms);
    printf("  Python/NumPy:   19.3 ms (from earlier test)\n");
    printf("  Speedup:        %.0fx faster! 🚀\n\n", 19.3/ms);

    printf("GPU launch config:\n");
    printf("  Threads: 16×16 = 256 per block\n");
    printf("  Blocks: %d×%d×3 = %d blocks\n", (out_w+15)/16, (out_h+15)/16, ((out_w+15)/16)*((out_h+15)/16)*3);
    printf("  Total threads: %d\n", ((out_w+15)/16)*((out_h+15)/16)*3*256);
    printf("  Each thread: processes ONE pixel in ONE channel\n\n");

    // Verify values are in range
    float min_val = 1.0f, max_val = 0.0f;
    for (int i = 0; i < out_size; i++) {
        if (h_output[i] < min_val) min_val = h_output[i];
        if (h_output[i] > max_val) max_val = h_output[i];
    }
    printf("Output range: [%.3f, %.3f] ✓ (should be 0-1)\n", min_val, max_val);

    free(h_input); free(h_output);
    cudaFree(d_input); cudaFree(d_output);
    return 0;
}


Writing preprocess.cu


In [ ]:
!nvcc preprocess.cu -o preprocess && ./preprocess


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
=== CUDA Image Preprocessing Kernel ===

Input:  1080×810×3 (uint8, HWC, 0-255)
Output: 3×640×640 (float32, CHW, 0-1)

Operations (all in ONE kernel):
  1. Resize: 1080×810 → 640×640 (nearest neighbor)
  2. Normalize: uint8 [0-255] → float32 [0-1]
  3. Layout: HWC → CHW (what neural networks expect)

Performance:
  CUDA kernel:    0.046 ms
  Python/NumPy:   19.3 ms (from earlier test)
  Speedup:        419x faster! 🚀

GPU launch config:
  Threads: 16×16 = 256 per block
  Blocks: 40×40×3 = 4800 blocks
  Total threads: 1228800
  Each thread: processes ONE pixel in ONE channel

Output range: [0.000, 1.000] ✓ (should be 0-1)


In [ ]:
%%writefile nms.cu
#include <stdio.h>
#include <stdlib.h>

// IoU = Intersection over Union (how much two boxes overlap)
__device__ float compute_iou(float *box1, float *box2) {
    // box format: [x1, y1, x2, y2]
    float x1 = fmaxf(box1[0], box2[0]);
    float y1 = fmaxf(box1[1], box2[1]);
    float x2 = fminf(box1[2], box2[2]);
    float y2 = fminf(box1[3], box2[3]);

    float intersection = fmaxf(0.0f, x2-x1) * fmaxf(0.0f, y2-y1);
    float area1 = (box1[2]-box1[0]) * (box1[3]-box1[1]);
    float area2 = (box2[2]-box2[0]) * (box2[3]-box2[1]);
    float union_area = area1 + area2 - intersection;

    return (union_area > 0) ? intersection / union_area : 0.0f;
}

// NMS kernel: for each box, check if a higher-confidence box overlaps it
// If yes → suppress (mark as removed)
__global__ void nms_kernel(float *boxes, float *scores, int *keep, int num_boxes,
                            float iou_threshold) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= num_boxes) return;

    // If already suppressed, skip
    if (keep[i] == 0) return;

    // Check against all higher-confidence boxes
    for (int j = 0; j < i; j++) {
        if (keep[j] == 0) continue;  // already suppressed

        float iou = compute_iou(&boxes[i*4], &boxes[j*4]);
        if (iou > iou_threshold) {
            keep[i] = 0;  // suppress this box (j has higher confidence)
            return;
        }
    }
}

// Sort boxes by confidence (descending) — simple CPU sort for demo
void sort_by_confidence(float *boxes, float *scores, int n) {
    for (int i = 0; i < n-1; i++)
        for (int j = i+1; j < n; j++)
            if (scores[j] > scores[i]) {
                // Swap scores
                float tmp = scores[i]; scores[i] = scores[j]; scores[j] = tmp;
                // Swap boxes
                for (int k = 0; k < 4; k++) {
                    tmp = boxes[i*4+k]; boxes[i*4+k] = boxes[j*4+k]; boxes[j*4+k] = tmp;
                }
            }
}

int main() {
    printf("=== CUDA NMS (Non-Maximum Suppression) ===\n\n");

    // Simulate YOLO output: many overlapping boxes for the same object
    // Format: [x1, y1, x2, y2] for each box
    int num_boxes = 10;
    float h_boxes[] = {
        // Bus detections (overlapping)
        100, 200, 400, 500,   // box 0: bus (best)
        105, 205, 395, 495,   // box 1: bus (slightly shifted)
        110, 210, 390, 490,   // box 2: bus (more shifted)
        95,  195, 405, 505,   // box 3: bus (slightly bigger)
        // Person detections (overlapping)
        500, 300, 600, 700,   // box 4: person (best)
        505, 305, 595, 695,   // box 5: person (shifted)
        510, 310, 590, 690,   // box 6: person (more shifted)
        // Separate detections (no overlap)
        50,  50,  150, 150,   // box 7: traffic light
        700, 100, 800, 200,   // box 8: sign
        300, 600, 450, 750,   // box 9: dog
    };
    float h_scores[] = {0.95, 0.90, 0.85, 0.80, 0.92, 0.88, 0.82, 0.75, 0.70, 0.65};

    printf("Before NMS: %d boxes\n", num_boxes);
    printf("%-6s %-30s %s\n", "Box", "Coordinates [x1,y1,x2,y2]", "Confidence");
    printf("------------------------------------------------------\n");
    for (int i = 0; i < num_boxes; i++)
        printf("%-6d [%3.0f, %3.0f, %3.0f, %3.0f]         %.2f\n",
               i, h_boxes[i*4], h_boxes[i*4+1], h_boxes[i*4+2], h_boxes[i*4+3], h_scores[i]);

    // Sort by confidence (highest first)
    sort_by_confidence(h_boxes, h_scores, num_boxes);

    // GPU memory
    float *d_boxes, *d_scores;
    int *d_keep;
    cudaMalloc(&d_boxes, num_boxes*4*sizeof(float));
    cudaMalloc(&d_scores, num_boxes*sizeof(float));
    cudaMalloc(&d_keep, num_boxes*sizeof(int));

    cudaMemcpy(d_boxes, h_boxes, num_boxes*4*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_scores, h_scores, num_boxes*sizeof(float), cudaMemcpyHostToDevice);

    // Initialize all as "keep"
    int h_keep[10] = {1,1,1,1,1,1,1,1,1,1};
    cudaMemcpy(d_keep, h_keep, num_boxes*sizeof(int), cudaMemcpyHostToDevice);

    // Run NMS
    float iou_threshold = 0.5f;  // suppress if overlap > 50%

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    cudaEventRecord(start);
    nms_kernel<<<(num_boxes+255)/256, 256>>>(d_boxes, d_scores, d_keep, num_boxes, iou_threshold);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    // Get results
    cudaMemcpy(h_keep, d_keep, num_boxes*sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nAfter NMS (IoU threshold: %.1f):\n", iou_threshold);
    printf("------------------------------------------------------\n");
    int kept = 0;
    for (int i = 0; i < num_boxes; i++) {
        if (h_keep[i]) {
            printf("✅ KEEP  box %d: [%3.0f, %3.0f, %3.0f, %3.0f] conf=%.2f\n",
                   i, h_boxes[i*4], h_boxes[i*4+1], h_boxes[i*4+2], h_boxes[i*4+3], h_scores[i]);
            kept++;
        } else {
            printf("❌ REMOVE box %d: [%3.0f, %3.0f, %3.0f, %3.0f] conf=%.2f (overlaps with better box)\n",
                   i, h_boxes[i*4], h_boxes[i*4+1], h_boxes[i*4+2], h_boxes[i*4+3], h_scores[i]);
        }
    }

    printf("\nResult: %d boxes → %d boxes (removed %d duplicates)\n", num_boxes, kept, num_boxes-kept);
    printf("NMS time: %.4f ms\n", ms);
    printf("\nWhat NMS did:\n");
    printf("  4 bus boxes (overlapping) → kept 1 best (0.95)\n");
    printf("  3 person boxes (overlapping) → kept 1 best (0.92)\n");
    printf("  3 separate boxes → kept all (no overlap)\n");

    printf("\n\nHow IoU works:\n");
    printf("  ┌──────────┐\n");
    printf("  │  Box A   │\n");
    printf("  │    ┌─────┼────┐\n");
    printf("  │    │INTER│    │\n");
    printf("  └────┼─────┘    │\n");
    printf("       │  Box B   │\n");
    printf("       └──────────┘\n");
    printf("  IoU = INTERSECTION area / UNION area\n");
    printf("  IoU > 0.5 → \"these are probably the same object\" → remove the weaker one\n");

    cudaFree(d_boxes); cudaFree(d_scores); cudaFree(d_keep);
    return 0;
}


Writing nms.cu


In [ ]:
!nvcc nms.cu -o nms && ./nms


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
=== CUDA NMS (Non-Maximum Suppression) ===

Before NMS: 10 boxes
Box    Coordinates [x1,y1,x2,y2]      Confidence
------------------------------------------------------
0      [100, 200, 400, 500]         0.95
1      [105, 205, 395, 495]         0.90
2      [110, 210, 390, 490]         0.85
3      [ 95, 195, 405, 505]         0.80
4      [500, 300, 600, 700]         0.92
5      [505, 305, 595, 695]         0.88
6      [510, 310, 590, 690]         0.82
7      [ 50,  50, 150, 150]         0.75
8      [700, 100, 800, 200]         0.70
9      [300, 600, 450, 750]         0.65

After NMS (IoU threshold: 0.5):
------------------------------------------------------
✅ KEEP  box 0: [100, 200, 400, 500] conf=0.95
✅ KEEP  box 1: [500, 300, 600, 700] conf=0.92
❌ REMOVE box 2: [105, 205, 395, 495] conf=0.90 (overla

In [ ]:
# Export YOLOv8 to TensorRT
from ultralytics import YOLO
import time

model = YOLO('yolov8n.pt')

# Export to TensorRT (this takes 1-2 minutes)
print("Exporting to TensorRT (FP16 precision)...")
model.export(format='engine', half=True)  # FP16 for speed
print("✓ Export complete!")


Exporting to TensorRT (FP16 precision)...
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 238ms
Prepared 4 packages in 4.30s
Installed 4 packages in 273ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 5.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for upda

In [ ]:
# Load TensorRT model and benchmark
model_trt = YOLO('yolov8n.engine')

# Warmup
for _ in range(5):
    model_trt('test_image.jpg', verbose=False)

# Benchmark TensorRT
times = []
for _ in range(20):
    start = time.time()
    results = model_trt('test_image.jpg', verbose=False)
    times.append((time.time() - start) * 1000)

trt_avg = sum(times) / len(times)

# Benchmark original PyTorch
model_pt = YOLO('yolov8n.pt')
for _ in range(5):
    model_pt('test_image.jpg', verbose=False)

times_pt = []
for _ in range(20):
    start = time.time()
    results_pt = model_pt('test_image.jpg', verbose=False)
    times_pt.append((time.time() - start) * 1000)

pt_avg = sum(times_pt) / len(times_pt)

# Compare
print(f"\n{'='*50}")
print(f"YOLO INFERENCE BENCHMARK")
print(f"{'='*50}")
print(f"PyTorch (FP32):    {pt_avg:.1f} ms  ({1000/pt_avg:.0f} FPS)")
print(f"TensorRT (FP16):   {trt_avg:.1f} ms  ({1000/trt_avg:.0f} FPS)")
print(f"Speedup:           {pt_avg/trt_avg:.1f}x faster!")
print(f"{'='*50}")
print(f"\nFull pipeline with our CUDA kernels:")
print(f"  Preprocess (CUDA):  0.05 ms (was 19.3ms in Python)")
print(f"  Inference (TRT):    {trt_avg:.1f} ms (was 168ms in PyTorch)")
print(f"  NMS (CUDA):         ~1 ms (was 72ms in Python)")
print(f"  TOTAL:              ~{trt_avg+1.05:.0f} ms = {1000/(trt_avg+1.05):.0f} FPS 🚀")
